# Deep Clustering (SimCLR / BYOL) для СЭМ-изображений

### Шаг 1: Подключите GPU
`Среда выполнения` -> `Сменить среду выполнения` -> T4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === Код из GitHub + данные из Drive ===
import os

# 1. Клонируем / обновляем код
if os.path.exists('/content/diploma_project/.git'):
    !cd /content/diploma_project && git pull
else:
    !git clone https://github.com/daniilctrl/sem-image-analysis.git /content/diploma_project

# 2. Распаковываем данные из zip (только если ещё не распакованы)
if not os.path.exists('/content/diploma_project/data/processed/tiles_metadata.csv'):
    !cp "/content/drive/MyDrive/diploma_data/processed_data_v2.zip" /content/
    !unzip -qo /content/processed_data_v2.zip -d /content/_data_tmp/
    !mkdir -p /content/diploma_project/data/processed
    !cp -r /content/_data_tmp/* /content/diploma_project/data/processed/
    !rm -rf /content/_data_tmp /content/processed_data_v2.zip
    print(f"Data extracted: {len(os.listdir('/content/diploma_project/data/processed'))} files")
else:
    print('Data already in place.')

!pip install -q umap-learn

---
## Обучение
Выберите **ОДНУ** из ячеек:
- **3a** — SimCLR с нуля
- **3b** — SimCLR продолжение (resume)
- **3c** — SimCLR v2 (batch=128, temp=0.2)
- **4** — BYOL

In [ ]:
# === 3a: SimCLR — первый запуск ===
!python /content/diploma_project/src/models/deep_clustering/train.py \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_checkpoints/ \
    --epochs 50 \
    --batch_size 64 \
    --workers 2

In [ ]:
# === 3b: SimCLR — продолжение после обрыва ===
LAST_EPOCH = 29  # <-- номер последнего чекпоинта
REMAINING = 50 - LAST_EPOCH

!python /content/diploma_project/src/models/deep_clustering/train.py \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_checkpoints/ \
    --epochs {REMAINING} \
    --batch_size 64 \
    --workers 2 \
    --resume /content/drive/MyDrive/diploma_checkpoints/simclr_resnet50_epoch_{LAST_EPOCH}.pth \
    --start_epoch {LAST_EPOCH}

In [ ]:
# === 3c: SimCLR v2 — агрессивные параметры ===
!python /content/diploma_project/src/models/deep_clustering/train.py \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_checkpoints_v2/ \
    --epochs 30 \
    --batch_size 128 \
    --temperature 0.2 \
    --workers 2

In [ ]:
# === 4: BYOL ===
!python /content/diploma_project/src/models/deep_clustering/train_byol.py \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_checkpoints_byol/ \
    --epochs 30 \
    --batch_size 64 \
    --workers 2

---
## Извлечение эмбеддингов и сравнение с Baseline

In [ ]:
# Укажи путь к лучшему чекпоинту:
CHECKPOINT = '/content/drive/MyDrive/diploma_checkpoints/simclr_resnet50_epoch_29.pth'
MODEL_TYPE = 'simclr'  # 'simclr' или 'byol'

# CHECKPOINT = '/content/drive/MyDrive/diploma_checkpoints_v2/simclr_resnet50_epoch_30.pth'
# CHECKPOINT = '/content/drive/MyDrive/diploma_checkpoints_byol/byol_resnet50_epoch_30.pth'
# MODEL_TYPE = 'byol'

!python /content/diploma_project/src/models/deep_clustering/extract_simclr_embeddings.py \
    --checkpoint {CHECKPOINT} \
    --model_type {MODEL_TYPE} \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_results/ \
    --batch_size 128 \
    --workers 2

In [ ]:
from IPython.display import Image, display
display(Image('/content/drive/MyDrive/diploma_results/baseline_vs_simclr_umap.png'))